# Stroke Prediction: Data Visualization & Preprocessing
Dataset: `healthcare-dataset-stroke-data.csv` (Kaggle) | Target: `stroke`

## Part 1: Data Loading & Initial Inspection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_style('whitegrid')
df = pd.read_csv('healthcare-dataset-stroke-data.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## Part 2: Exploratory Data Analysis & Visualization

In [ ]:
# 1. Class distribution
plt.figure(figsize=(5, 4))
sns.countplot(x='stroke', data=df)
plt.title('Stroke Class Distribution')
plt.show()
print(df['stroke'].value_counts())

In [ ]:
# 2. Countplots with hue='stroke'
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ['gender', 'work_type', 'smoking_status']):
    sns.countplot(x=col, hue='stroke', data=df, ax=ax)
    ax.set_title(f'{col} vs stroke')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Drop the single 'Other' gender record
print(df['gender'].value_counts())
df = df[df['gender'] != 'Other']
print(df['gender'].value_counts())

In [ ]:
# 4. Histograms / KDE
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ['age', 'avg_glucose_level', 'bmi']):
    sns.histplot(df[col].dropna(), kde=True, ax=ax)
    ax.set_title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

In [ ]:
# 5. Correlation heatmap
num_cols = ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi', 'stroke']
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

## Part 3: Data Preprocessing (Cleaning)

In [ ]:
# 1. Missing values
print(df.isnull().sum())

# 2. Fill bmi with median (robust to outliers)
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

# 3. Verify
print('\nAfter filling:')
print(df.isnull().sum())

In [ ]:
# 4. Duplicates
print('Duplicates:', df.duplicated().sum())
df = df.drop_duplicates()

# 5. Drop id
df = df.drop(columns=['id'])
df.shape

In [ ]:
# 6. Boxplot for avg_glucose_level + outlier removal (IQR)
plt.figure(figsize=(8, 3))
sns.boxplot(x=df['avg_glucose_level'])
plt.title('avg_glucose_level (before)')
plt.show()

Q1 = df['avg_glucose_level'].quantile(0.25)
Q3 = df['avg_glucose_level'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
print('Bounds:', lower, upper)
print('Rows before:', len(df))
df = df[(df['avg_glucose_level'] >= lower) & (df['avg_glucose_level'] <= upper)]
print('Rows after:', len(df))

plt.figure(figsize=(8, 3))
sns.boxplot(x=df['avg_glucose_level'])
plt.title('avg_glucose_level (after)')
plt.show()

> Note: stroke patients often have high glucose, so IQR removal may drop some real positive cases. Check `df['stroke'].value_counts()` after removal; if too many positives are lost, skip this step.

## Part 4: Feature Transformation

In [ ]:
# 2. Reset index
df = df.reset_index(drop=True)

# 1. Encode binary categoricals
df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
df['ever_married'] = df['ever_married'].map({'No': 0, 'Yes': 1})
df['Residence_type'] = df['Residence_type'].map({'Rural': 0, 'Urban': 1})

# 3. One-hot encode multi-class categoricals
df = pd.get_dummies(df, columns=['work_type', 'smoking_status'], drop_first=True, dtype=int)
df.head()

## Part 5: Data Splitting

In [ ]:
X = df.drop(columns=['stroke'])
y = df['stroke']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('y_train:', y_train.shape)
print('y_test :', y_test.shape)

## Part 6: Feature Scaling

In [ ]:
scale_cols = ['age', 'avg_glucose_level', 'bmi']
scaler = StandardScaler()

X_train = X_train.copy()
X_test = X_test.copy()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])  # fit on train only
X_test[scale_cols] = scaler.transform(X_test[scale_cols])
X_train[scale_cols].describe()

## Part 7: Handling Class Imbalance
Applied on the **training set only** to avoid data leakage.

In [ ]:
# 1. Check distribution
print('Before balancing:')
print(y_train.value_counts())

# 2. SMOTE (needs: pip install imbalanced-learn); fallback = random oversampling
try:
    from imblearn.over_sampling import SMOTE
    X_train_bal, y_train_bal = SMOTE(random_state=42).fit_resample(X_train, y_train)
    print('\nUsed SMOTE')
except ImportError:
    from sklearn.utils import resample
    train = pd.concat([X_train, y_train], axis=1)
    major = train[train['stroke'] == 0]
    minor = train[train['stroke'] == 1]
    minor_up = resample(minor, replace=True, n_samples=len(major), random_state=42)
    train_bal = pd.concat([major, minor_up]).sample(frac=1, random_state=42)
    X_train_bal, y_train_bal = train_bal.drop(columns=['stroke']), train_bal['stroke']
    print('\nimblearn not installed: used random oversampling')

# 3. Verify
print('\nAfter balancing:')
print(y_train_bal.value_counts())
sns.countplot(x=y_train_bal)
plt.title('Balanced training set')
plt.show()

## Conclusion
### High-Risk Patient Persona
**Mr. Ahmed, 72 years old, male, married.** He has hypertension and heart disease, an average glucose level above 150 mg/dL, and a BMI around 30 (overweight/obese). He is a former or current smoker, works in the private sector (or is retired), and lives in an urban area.

### Summary of Findings
- The dataset is highly imbalanced: only about 5% of patients had a stroke.
- **Age** is the strongest predictor; stroke cases cluster in older ages.
- **Hypertension, heart disease and high glucose** show positive correlation with stroke.
- `bmi` had missing values (filled with the median); `id` and the single 'Other' gender record were dropped.
- Categorical features were encoded, data split 80/20 (random_state=42), numerical features standardized (fit on train only), and the training set balanced with SMOTE/oversampling.
- The data is now ready for modeling (Logistic Regression, Random Forest, etc.), evaluated with recall/F1/ROC-AUC rather than accuracy alone.

*Verify the numbers above against your own outputs after running the notebook.*